In [4]:
"""Calculate exact memory usage for the SigLIP OOM scenario."""
import numpy as np
from math import comb
from transformers import AutoProcessor, AutoModel
from src.fixlip import FIxLIP

# ── 1. Load SigLIP model and inspect config for player counts ────────────
model = AutoModel.from_pretrained("google/siglip-base-patch16-224")
processor = AutoProcessor.from_pretrained("google/siglip-base-patch16-224")
input_text = "black dog next to a yellow hydrant"

# Determine player counts the same way the factory does
vision_config = model.config.vision_config
image_size = vision_config.image_size       # 224
patch_size = vision_config.patch_size       # 16
grid_size = image_size // patch_size        # 14
n_img = grid_size ** 2                      # 196

# Text: count players via tokenizer (strip BOS/EOS for CLIP; count non-pad for SigLIP)
inputs = processor(text=[input_text], return_tensors="pt", padding="max_length", max_length=64)
n_txt = (inputs["input_ids"][0] != 1).count_nonzero().item()  # SigLIP: non-padding tokens
n_total = n_img + n_txt

print(f"Model: SigLIP base patch16-224")
print(f"image_size={image_size}, patch_size={patch_size}, grid_size={grid_size}")
print(f"\nPlayer counts:  image={n_img}  text={n_txt}  total={n_total}")

# ── 2. Interaction count ────────────────────────────────────────────────
def n_int(n_players, max_order):
    return sum(comb(n_players, k) for k in range(max_order + 1))

n_int_m1 = n_int(n_total, 1)
n_int_m2 = n_int(n_total, 2)
print(f"\nInteractions (max_order=1): {n_int_m1:,}")
print(f"Interactions (max_order=2): {n_int_m2:,}")

# ── 3. Memory table ─────────────────────────────────────────────────────
fixlip_m2 = FIxLIP(n_players_image=n_img, n_players_text=n_txt,
                   max_order=2, p=0.5, mode="banzhaf", random_state=42)

GiB = 1024**3
print(f"\n{'Budget':>10s}  {'split':>12s}  {'coalitions':>10s}  {'X(N×P)':>12s}  {'+W@X':>12s}  {'+XᵀWX':>12s}  {'peak':>12s}  {'GPU':>8s}  {'total':>8s}")
print(f"{'─'*10}  {'─'*12}  {'─'*10}  {'─'*12}  {'─'*12}  {'─'*12}  {'─'*12}  {'─'*8}  {'─'*8}")

for exp in [10, 12, 14, 16, 17, 18, 19]:
    budget = 2 ** exp
    bi, bt = fixlip_m2.split_budget(budget)
    n_coal = bi * bt
    
    for label, n_i in [("orig max_order=2", n_int_m2), ("chunked max_order=1", n_int_m1)]:
        X_bytes = n_coal * n_i * 8          # regression_matrix (float64)
        WX_bytes = n_coal * n_i * 8          # W @ X (float64)
        XtWX_bytes = n_i * n_i * 8           # X^T W X accumulator
        peak = X_bytes + WX_bytes + XtWX_bytes
        
        gpu_mem = 4.0  # SigLIP model on GPU ~4 GiB
        print(f"2^{exp:<2d} ({budget:>6d})  {bi:>5d}×{bt:<3d}  {n_coal:>10,d}  "
              f"{X_bytes/GiB:>8.2f} GiB  {WX_bytes/GiB:>8.2f} GiB  "
              f"{XtWX_bytes/GiB:>8.2f} GiB  {peak/GiB:>8.2f} GiB  "
              f"{gpu_mem:.1f} GiB  {peak/GiB + gpu_mem:>6.1f} GiB  ({label})")

    print()


Model: SigLIP base patch16-224
image_size=224, patch_size=16, grid_size=14

Player counts:  image=196  text=11  total=207

Interactions (max_order=1): 208
Interactions (max_order=2): 21,529

    Budget         split  coalitions        X(N×P)          +W@X         +XᵀWX          peak       GPU     total
──────────  ────────────  ──────────  ────────────  ────────────  ────────────  ────────────  ────────  ────────
2^10 (  1024)    256×4         1,024      0.16 GiB      0.16 GiB      3.45 GiB      3.78 GiB  4.0 GiB     7.8 GiB  (orig max_order=2)
2^10 (  1024)    256×4         1,024      0.00 GiB      0.00 GiB      0.00 GiB      0.00 GiB  4.0 GiB     4.0 GiB  (chunked max_order=1)

2^12 (  4096)   1024×4         4,096      0.66 GiB      0.66 GiB      3.45 GiB      4.77 GiB  4.0 GiB     8.8 GiB  (orig max_order=2)
2^12 (  4096)   1024×4         4,096      0.01 GiB      0.01 GiB      0.00 GiB      0.01 GiB  4.0 GiB     4.0 GiB  (chunked max_order=1)

2^14 ( 16384)   2048×8        16,384   